# 03. Budget-Constrained Optimization & Knapsack Formulation
**Team Member**: Optimization Analyst (Member 3)
**Objective**: Formulate the 0-1 Knapsack problem using ITE estimates, solve via SciPy MILP, and benchmark against heuristic targeting policies.

In [1]:
import pandas as pd
import numpy as np
from src.data.synthetic import generate_synthetic_data
from src.models.dml_estimator import DoubleMLEstimator
from src.optimization.knapsack import solve_knapsack_discount_allocation
from src.optimization.policy_allocator import compare_allocation_policies

# Generate data and obtain ITE estimates
df = generate_synthetic_data(n_samples=5000, random_state=42)
features = ["Age", "PriorSpend", "VisitFrequency", "LoyaltyScore"]
dml = DoubleMLEstimator(n_splits=3, nuisance_type="random_forest", backend="native").fit(df[features], df["Discount"], df["Sales"])
df["Estimated_ITE"] = dml.ite_estimates_
print("ITE estimation ready for optimization.")

[2026-10-21 15:54:55] [INFO] [DoubleMLEstimator]: Fitting DML Estimator using backend 'native' and 'random_forest' nuisances...


[2026-10-21 15:54:56] [INFO] [DoubleMLEstimator]: Native DML fit complete: ATE = 31.092 +/- 1.130


ITE estimation ready for optimization.


In [2]:
# Solve 0-1 Knapsack with a $5,000 budget
knapsack_res = solve_knapsack_discount_allocation(
    ite_estimates=df["Estimated_ITE"].values,
    item_prices=df["EstimatedItemPrice"].values,
    budget=5000.0,
    discount_rate=0.15,
    method="milp"
)
print(f"Targeted Customers: {knapsack_res['n_targeted']}")
print(f"Total Cost: ${knapsack_res['total_cost']:.2f}")
print(f"Expected Incremental Gain: ${knapsack_res['total_incremental_value']:.2f}")
print(f"ROI: {knapsack_res['estimated_roi']:.2f}x")

[2026-10-21 15:54:56] [INFO] [KnapsackOptimizer]: Setting up Knapsack for 5000 candidates with budget $5,000.00...


[2026-10-21 15:54:58] [INFO] [KnapsackOptimizer]: Optimization complete: Targeted 219 customers | Cost: $5,000.00 / $5,000.00 | Incremental Value: $159,747.77 | ROI: 31.95x


Targeted Customers: 219
Total Cost: $5000.00
Expected Incremental Gain: $159747.77
ROI: 31.95x


In [3]:
# Compare Causal Optimal vs Heuristic vs Random
policy_comparison = compare_allocation_policies(df, df["Estimated_ITE"].values, budget=5000.0)
policy_comparison

[2026-10-21 15:54:58] [INFO] [KnapsackOptimizer]: Setting up Knapsack for 5000 candidates with budget $5,000.00...


[2026-10-21 15:54:59] [INFO] [KnapsackOptimizer]: Optimization complete: Targeted 219 customers | Cost: $5,000.00 / $5,000.00 | Incremental Value: $159,747.77 | ROI: 31.95x


,Policy,Targeted_Customers,Budget_Spent,Incremental_Gain,ROI,Lift_vs_Random (%)
0,Causal DML Optimal,219,5000.00,159747.77,31.95,52.1
1,High-Spender Heuristic,158,4995.27,92538.72,18.53,-11.9
2,Random Targeting,269,4999.51,105058.45,21.01,0.0
